# TOP GENE SET-BASED TEST PLOT 
 
This script generates plots displaying the top genes and the most significant test results from the gene set-based tests, defined as those with p-values less than 1 × 10⁻⁴. Both Cauchy and non-Cauchy p-values are shown.
- Plot 1 : Top non-Cauchy SKAT-O p-value plots 
- Plot 2 :Top Cauchy SKAT-O p-value plots 

In [ ]:
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install("biomaRt")

In [ ]:
# Load the package
library(tidyverse)
library(ggplot2)
library(data.table)
library(purrr)
# Install these packages 
library(janitor)
library(flextable)
library(officer)
library(biomaRt)
library(patchwork)

In [ ]:
# Connect to Ensembl
mart <- useEnsembl(
    biomart = "genes",
    dataset = "hsapiens_gene_ensembl",
    mirror = "useast"
)

In [ ]:
# Function: Formats the gene set-based rare variant assoication test results 
process_results <- function(results) { 
    results %>% 
    # filter(Group != "Cauchy") %>% 
    # inner_join(gene_symbols, by = "Region") %>% 
    mutate(max_MAF = as.character(max_MAF)) %>% 
    mutate(negative_log_p_values = -log10(Pvalue)) %>% 
    # mutate(Chromosome = gsub("chr", "", Chromosome)) %>% 
    mutate(Chromosome = as.numeric(Chromosome)) %>% 
    mutate(Region = as.factor(Region)) %>% 
    mutate(Region_factor = fct_reorder(Region, Chromosome))
  }

In [ ]:
# Load data from workspace bucket  
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

files <- c()
stats <- c() 

system("mkdir -p local_dir") 
phenotypes <- c("aml_phenotype",
                "aml_biosample_age", 
                "aml_diagnosis_age")

for (pheno in phenotypes){
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test:", pheno))
        print(paste0("Loading chromosome:", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/HC_analysis/EUR_", 
            pheno, 
            "_high_confidence_chr",
            chr,
            " local_dir"
        )

        system(cmd)

        path <- paste0("local_dir/EUR_",
                       pheno,
                       "_high_confidence_chr", chr)
        f <- fread(path)
        
        f$Test <- pheno
        f$Chromosome <- chr
        
        processed <- process_results(f)

        gene_info <- getBM(
          attributes = c("ensembl_gene_id", "hgnc_symbol"),
          filters = "ensembl_gene_id",
          values = unique(processed$Region),
          mart = mart
        )

        gene_info <- 
        rename(gene_info, 
           "Region" = "ensembl_gene_id", 
           "Symbol" = "hgnc_symbol" )

        processed <- full_join(gene_info, processed, by ="Region")

        files[[chr]] <- processed
    }
    
    stat <- do.call(rbind, files)
    
    print("Check Groups Counts")
    print(table(stat$Group))
    
    stat <- filter(stat,, Pvalue < 1e-4)
    stats[[pheno]] <- stat
}


In [ ]:
# Load data from workspace bucket  
system("mkdir -p local_dir") 
covariates <- c("_deprivation_index",
                "_deprivation")

for (cov in covariates){
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test:", pheno))
        print(paste0("Loading chromosome:", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/HC_analysis/EUR_", 
            "aml_phenotype", 
            "_high_confidence_chr",
            chr, 
            cov, 
            " local_dir"
        )

        system(cmd)

        path <- paste0("local_dir/EUR_",
                       "aml_phenotype",
                       "_high_confidence_chr", chr, cov)
        f <- fread(path)
        
        f$Test <- cov
        f$Chromosome <- chr
        
        processed <- process_results(f)

        gene_info <- getBM(
          attributes = c("ensembl_gene_id", "hgnc_symbol"),
          filters = "ensembl_gene_id",
          values = unique(processed$Region),
          mart = mart
        )

        gene_info <- 
        rename(gene_info, 
           "Region" = "ensembl_gene_id", 
           "Symbol" = "hgnc_symbol" )

        processed <- full_join(gene_info, processed, by ="Region")

        files[[chr]] <- processed
    }
    
    stat <- do.call(rbind, files)
    
    print("Check Groups Counts")
    print(table(stat$Group))
    
    stat <- filter(stat,, Pvalue < 1e-4)
    stats[[cov]] <- stat
}

stats <- do.call(rbind, stats)

In [ ]:
# Set encode colors for the plots 
color_values <- c("1e-04" = "#172869FF",
                  "0.001" = "#F7E690FF",
                  "0.01" = "#F5D000FF")

In [ ]:
 paletteer::paletteer_d("LaCroixColoR::Lemon")

In [ ]:
options(repr.plot.height = 9, repr.plot.width = 20)

In [ ]:
# ========================================================================================================
# Plot 1 : Top genes skato-o p-values 
# ========================================================================================================
    df <- stats %>% filter(Group != "Cauchy") 
  
  df$Chromosome <- as.numeric(as.character(df$Chromosome))

  df$Test <- factor(df$Test,
                      levels = c(
                        "_deprivation_index",
                        "_deprivation",
                        "aml_phenotype",
                        "aml_biosample_age",
                        "aml_diagnosis_age"
                      ))
  
  # Proper Manhattan-style ordering
  df <- df %>%
    arrange(negative_log_p_values) %>%
    group_by(Test) %>% 
    mutate(region_index = row_number()) %>%
    ungroup() 
  
  # Top hits
  top_values <- df %>%
    arrange(-negative_log_p_values) %>%
    slice(1:100)
  
  plot1 <- ggplot(df, aes(x = region_index, 
                         y = negative_log_p_values)) +
    
    geom_jitter(
      aes(color = max_MAF, shape = Group),
      height = 0,
      width = 10,
      size = 4
    ) +

  geom_hline(yintercept = -log10((0.05/(50519 + 53046 + 54188))),
             linetype = "dashed",
             color = "red",
             linewidth = 1.5) + 
    
    scale_shape_discrete(labels = c("LoF", "Missense", "LoF+Missense")) +
    scale_color_manual(values = color_values,
                       labels = c("1 x 10²", "1 x 10³", "1 x 10⁴")) +
    
    labs(y = "-log10(p-value)",
         shape = "Annotation",
         color = "MAF") +
    
    theme_bw() +
    theme(
      axis.title.x = element_blank(),
      legend.key = element_rect(fill = "white"),
      legend.text = element_text(size = 16, face = "italic"),
      legend.title = element_text(size = 20, face = 'bold'),
      plot.title = element_text(size = 22, face ='bold'),
      axis.title.y = element_text(size = 24, face ='bold', margin = margin(r = 10)),
      axis.text.y = element_text(size = 24),
      panel.grid.major.x = element_blank(),
      panel.grid.minor.x = element_blank(),
      axis.text.x = element_blank(),
      axis.ticks.x = element_blank(),
      strip.background = element_blank(),
      strip.text = element_text(size = 18, face ='bold'),
      panel.border = element_rect(colour = "black", fill = NA, linewidth = 4) 
    ) +
    
    guides(
      size = "none",
      color = guide_legend(override.aes = list(size = 5)),
      shape = guide_legend(override.aes = list(size = 5))
    ) +
    
    geom_text(
      data = top_values,
      aes(x = region_index, 
          y = negative_log_p_values, 
          label = Symbol),
      check_overlap = TRUE,
      vjust = -1,
      hjust = 1,
      size = 8
    ) +
    
    facet_grid(cols = vars(Test),
                  labeller = labeller(Test = c(
                   "_deprivation" = "Deprivation Components",
                   "_deprivation_index" = "Deprivation Index",
                    "aml_biosample_age" = "Age at Collection",
                    "aml_diagnosis_age" = "Age at Diagnosis",
                    "aml_phenotype" = "Age at Last EHR"
                 ))) +
    ylim(4,12.5)

In [ ]:
# ========================================================================================================
# Plot 2 : Top genes cauchy p-values 
# ========================================================================================================
    df <- stats %>% 
        filter(Group == "Cauchy") %>% 
        mutate(max_MAF = "0.01")
  
  df$Chromosome <- as.numeric(as.character(df$Chromosome))

  df$Test <- factor(df$Test,
                      levels = c(
                        "_deprivation_index",
                        "_deprivation",
                        "aml_phenotype",
                        "aml_biosample_age",
                        "aml_diagnosis_age"
                      ))
  
  # Proper Manhattan-style ordering
  df <- df %>%
    arrange(negative_log_p_values) %>%
    mutate(region_index = row_number()) 
  
  # Top hits
  top_values <- df %>%
    arrange(-negative_log_p_values) %>%
    slice(1:30)
  
  plot2 <- ggplot(df, aes(x = region_index, 
                         y = negative_log_p_values)) +

    geom_point(
        aes(
            color = max_MAF, 
            shape = Group),
      size = 6
    ) +

  geom_hline(yintercept = -log10((0.05/18064)),
             linetype = "dashed",
             color = "red",
             linewidth = 1.5) + 
    
    labs(y = "-log10(p-value)",
        color = "Minor Allele Frequency", 
        shape = "Functional Group"
        ) +

    scale_color_manual(values = color_values,
                        labels = c("1 x 10²")) + 
    
    theme_bw() +
    theme(
      axis.title.x = element_blank(),
      legend.key = element_rect(fill = "white"),
      legend.text = element_text(size = 16, face = "italic"),
      legend.title = element_text(size = 10, face = 'bold'),
      plot.title = element_text(size = 22, face ='bold'),
      axis.title.y = element_text(size = 24, face ='bold', margin = margin(r = 10)),
      axis.text.y = element_text(size = 24),
      panel.grid.major.x = element_blank(),
      panel.grid.minor.x = element_blank(),
      axis.text.x = element_blank(),
      axis.ticks.x = element_blank(),
      strip.background = element_blank(),
      strip.text = element_text(size = 18, face ='bold'),
      panel.border = element_rect(colour = "black", fill = NA, linewidth = 4) 
    ) +
    
    guides(
      size = "none",
      shape = "none", 
      color = "none"
    ) +
    
    geom_text(
      data = top_values,
      aes(x = region_index, 
          y = negative_log_p_values, 
          label = Symbol),
      check_overlap = TRUE,
      vjust = 0,
      hjust = 1.2,
      size = 8
    ) +
    
    facet_grid(cols = vars(Test),
                  labeller = labeller(Test = c(
                   "_deprivation" = "Deprivation Components",
                   "_deprivation_index" = "Deprivation Index",
                    "aml_biosample_age" = "Age at Collection",
                    "aml_diagnosis_age" = "Age at Diagnosis",
                    "aml_phenotype" = "Age at Last EHR"
                 ))) + 
    ylim(4,12.5)

In [ ]:
# ========================================================================================================
# Save plot 
# ========================================================================================================
plot <- (plot1 / plot2) + 
    plot_annotation(tag_levels = 'A') &
    theme(plot.tag = element_text(size = 30, face = "bold"))

ggsave(
  "top_gene_set_based_tests.png",
  width = 22,
  height = 18,
  dpi = 300
)